<a href="https://colab.research.google.com/github/dkalchenko01/UA-Locations-NER-Deep-Learning/blob/stacked/notebooks/04_bigru_stacked.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stacked BiGRU · Whitespace BPE

**Control run:** frozen existing 32k Whitespace BPE tokenizer → randomly initialized trainable embedding 128 → 3-layer BiGRU h256/direction → BIO logits. No pretrained embeddings.

## 1. Setup / install

In [1]:
import os, sys, base64, subprocess
from pathlib import Path
from google.colab import userdata

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"
root = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")
repo_dir = Path(os.environ.get("UA_NER_REPO_DIR", root / "UA-Locations-NER-Deep-Learning"))

if not repo_dir.exists():
    token = userdata.get("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone", "-b", "stacked",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(repo_dir)], env=env, check=True)
    finally:
        del token, auth, env

# Check repo and required files
required = [
    "src/bigru_model.py",
    "src/bigru_helpers.py",
    "src/bigru_sweeps.py",
]

if not all((repo_dir / file).is_file() for file in required):
    subprocess.run([
        "git", "-C", str(repo_dir),
        "pull", "--ff-only", "origin", "stacked"
    ], check=True)

# Make src importable
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("Repository:", repo_dir)


Repository: /content/UA-Locations-NER-Deep-Learning


In [2]:
%pip install -q -r "$repo_dir/requirements.txt"

In [3]:
import gc, json, platform, uuid
from importlib.metadata import version
from time import perf_counter

import pandas as pd
import torch
from tokenizers import Tokenizer
import wandb

from src.bigru_model import StackedBiGRUTagger
from src.bigru_helpers import CLEAN_OPTIONS, backup_files, postprocessed_scores, export_token_logits_jsonl
from src.data_io import get_data_dir, load_manifest, load_data_splits, save_json, file_sha256
from src.dataset import LABEL_TO_ID, prepare_ner_records, prepare_inference_records, make_loader
from src.metrics import competition_scores
from src.preprocessing import clean_locations_v3
from src.training import set_seed, train_epoch, predict, save_checkpoint
from src.experiment_tracking import start_run, log_epoch, finish_run

assert torch.cuda.is_available(), "GPU required for full training: Runtime > Change runtime type > GPU"
device = torch.device("cuda")
print("PyTorch:", torch.__version__, ", GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.14.1+cu130 , GPU: Tesla T4


## 2. External services (W&B + Drive)

In [4]:
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"
os.environ["WANDB_DISABLE_CODE"] = "true"
os.environ["WANDB_CONSOLE"] = "off"
from google.colab import userdata
wandb.login(key=userdata.get("WANDB_API_KEY"), relogin=True)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: Currently logged in as: daldoshyna (daldoshyna-kyiv-school-of-economics) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


True

In [5]:
from google.colab import drive
drive.mount("/content/drive")
checkpoint_backup_root = Path("/content/drive/MyDrive/new_folder_that_diana_made_for_deeplearning_delete_it_later/UA-Locations-NER/stacked_bigru")

print("Checkpoint backup root:", checkpoint_backup_root)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint backup root: /content/drive/MyDrive/new_folder_that_diana_made_for_deeplearning_delete_it_later/UA-Locations-NER/stacked_bigru


## 3. Control configuration

In [6]:
config = {
    "model": "bigru", "architecture": "stacked", "embedding_dim": 128,
    "hidden_size": 256, "num_layers": 3, "dropout": 0.2,
    "seed": 17, "batch_size": 128, "learning_rate": 0.001, "epochs": 3,
    "gradient_clip": 1.0, "max_padded_tokens": 16384,
    "prepare_batch_size": 2000, "sources": ["uk", "ru"],
    "optimizer": "Adam", "tokenizer": "bpe_whitespace.json",
    "selection_metric": "uk_comp_f1", "label_to_id": LABEL_TO_ID,
    "max_sequence_length": None, "early_stopping": False,
    "environment": ENVIRONMENT, "validation_version": "v1",
    "metric_version": "kaggle_original_regex_v1",
    "postprocessing_version": "raw", "upload_checkpoint": False,
}
set_seed(config["seed"])
splits_dir = repo_dir / "data/splits"
tokenizer_path = repo_dir / "artifacts/tokenizers" / config["tokenizer"]
run_id = "bigru_ws_" + pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
run_dir = root / "outputs" / run_id
run_dir.mkdir(parents=True, exist_ok=False)
backup_dir = (checkpoint_backup_root / run_id)
backup_dir.mkdir(parents=True, exist_ok=True)
print(run_id)


bigru_ws_20261112T181358Z_c5b76509


In [7]:
provenance = {
    "run_id": run_id, "seed": config["seed"], "python": platform.python_version(),
    "git_revision": subprocess.run(["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
                                   capture_output=True, text=True).stdout.strip() or None,
    "git_status": subprocess.run(["git", "-C", str(repo_dir), "status", "--short"],
                                 capture_output=True, text=True).stdout.strip(),
    "packages": {name: version(name) for name in
                 ["torch", "tokenizers", "numpy", "pandas", "pyarrow", "tqdm", "wandb"]},
    "device": str(device), "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "tokenizer_sha256": file_sha256(tokenizer_path),
    "manifest_sha256": file_sha256(splits_dir / "validation_manifest.parquet"),
    "source_sha256": {path.name: file_sha256(path) for path in
                      sorted((repo_dir / "src").glob("*.py"))},
    "notebook_sha256": (
    file_sha256(repo_dir / "notebooks/04_bigru_stacked.ipynb")
    if (repo_dir / "notebooks/04_bigru_stacked.ipynb").is_file()
    else None
),
}
save_json(run_dir / "config.json", config)
save_json(run_dir / "provenance.json", provenance)
backup_files(backup_dir, run_dir/"config.json", run_dir/"provenance.json", tokenizer_path)


## 4. Tokenizer

In [8]:
tokenizer = Tokenizer.from_file(str(tokenizer_path))
tokenizer.no_padding()
tokenizer.no_truncation()

pad_id = tokenizer.token_to_id("[PAD]")

print("Whitespace BPE vocabulary:", tokenizer.get_vocab_size())

Whitespace BPE vocabulary: 32000


## 5. Fixed data splits & alignment diagnostics

This is the expensive CPU preprocessing stage; UK/RU splits are unchanged. Validation alignment failures remain in competition F1 but have no BIO loss. Stop if >5% alignment fails.

In [9]:
train_records, val_records, alignment = [], [], []

for source in config["sources"]:
    splits = load_data_splits(ENVIRONMENT, splits_dir, sources=(source,))

    for split, target in [("train", train_records), ("val", val_records)]:
        frame = splits[f"{source}_{split}"][["text", "loc_markers"]]

        records, skipped = prepare_ner_records(
            tokenizer, frame, source, split,
            batch_size=config["prepare_batch_size"]
        )

        target.extend(records)

        alignment.append({
            "source": source,
            "split": split,
            "total": len(frame),
            "alignment_failures": len(skipped)
        })

alignment = pd.DataFrame(alignment)
alignment["failure_rate"] = (
    alignment["alignment_failures"] / alignment["total"]
)

display(alignment)

loader_kw = {
    "pad_id": pad_id,
    "batch_size": config["batch_size"],
    "seed": config["seed"],
    "max_padded_tokens": config["max_padded_tokens"],
    "pin_memory": True
}

train_loader = make_loader(train_records, shuffle=True, **loader_kw)
val_loader = make_loader(val_records, **loader_kw)

print(f"Train: {len(train_records):,}, Validation: {len(val_records):,}")

Prepare uk/train:   0%|          | 0/403 [00:00<?, ?it/s]

Prepare uk/val:   0%|          | 0/101 [00:00<?, ?it/s]

Prepare ru/train:   0%|          | 0/50 [00:00<?, ?it/s]

Prepare ru/val:   0%|          | 0/25 [00:00<?, ?it/s]

,source,split,total,alignment_failures,failure_rate
0,uk,train,805983,1616,0.002005
1,uk,val,201498,375,0.001861
2,ru,train,100000,91,0.000910
3,ru,val,49369,38,0.000770


Train: 904,276, Validation: 250,867


## 6. Model / W&B run

In [10]:
os.environ["WANDB_DIR"] = str(run_dir)
run = start_run(
    model="bigru", config=config, name=run_id,
    validation_version=config["validation_version"], metric_version=config["metric_version"],
    postprocessing_version="raw", repo_dir=repo_dir,
    split_manifest_path=splits_dir / "validation_manifest.parquet",
    tokenizer_path=tokenizer_path, provenance=provenance,
    tags=["stacked", "bigru", "whitespace-bpe", "control"],
)
save_json(run_dir / "wandb_run.json", {
    "id": run.id, "url": run.url, "path": f"{run.entity}/{run.project}/{run.id}",
})
backup_files(backup_dir, run_dir/"wandb_run.json")
print(run.url)


https://app.forge.coreweave.com/daldoshyna-kyiv-school-of-economics/ua-location-ner/runs/qr0yxwnc


In [11]:
model_config = {
    "vocab_size": tokenizer.get_vocab_size(), "pad_id": pad_id,
    "embedding_dim": config["embedding_dim"], "hidden_size": config["hidden_size"],
    "num_layers": config["num_layers"], "dropout": config["dropout"],
    "num_labels": len(LABEL_TO_ID),
}
model = StackedBiGRUTagger(**model_config).to(device)
optimizer_cls = getattr(torch.optim, config["optimizer"])
optimizer_kwargs = {"lr": config["learning_rate"]}
if config["optimizer"] == "AdamW":
    optimizer_kwargs["weight_decay"] = config.get("weight_decay", 0.01)
optimizer = optimizer_cls(model.parameters(), **optimizer_kwargs)
save_json(run_dir / "model_config.json", model_config)
backup_files(backup_dir, run_dir / "model_config.json")
run.config.update({"model_config": model_config, "optimizer_effective": config["optimizer"]})
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


StackedBiGRUTagger(
  (embedding): Embedding(32000, 128, padding_idx=0)
  (gru): GRU(128, 256, num_layers=3, batch_first=True, dropout=0.2, bidirectional=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (classifier): Linear(in_features=512, out_features=3, bias=True)
)
Trainable parameters: 7055875


## 7. TRAIN — three epochs

Saves `epoch_01.pt`, `epoch_02.pt`, `epoch_03.pt`, `best.pt` locally and to Drive after each epoch.

In [12]:
history, best_score, best_epoch = [], float("-inf"), None
best_path = run_dir / "best.pt"
for epoch in range(1, config["epochs"] + 1):
    epoch_started = perf_counter()
    train_loader.batch_sampler.set_epoch(epoch)
    train_loss = train_epoch(model, train_loader, optimizer, device, config["gradient_clip"])
    predictions, val_losses = predict(model, val_loader, device, decoding="argmax")
    scores = competition_scores(val_records, predictions)
    clean_scores = competition_scores(val_records, predictions, cleaner=clean_locations_v3)
    result = {
        "epoch": epoch, "train_loss": train_loss,
        "epoch_seconds": perf_counter() - epoch_started,
        **{f"{source}_val_loss": value for source, value in val_losses.items()},
        **scores, **{f"clean_{key}": value for key, value in clean_scores.items()},
    }
    history.append(result)
    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    epoch_path = run_dir / f"epoch_{epoch:02d}.pt"
    save_checkpoint(epoch_path, model, optimizer, epoch, config, model_config,
                    scores, provenance)
    if scores[config["selection_metric"]] > best_score:
        best_score = scores[config["selection_metric"]]
        best_epoch = epoch
        save_checkpoint(best_path, model, optimizer, epoch, config,
                        model_config, scores, provenance)
        backup_files(backup_dir, best_path)
    backup_files(backup_dir, epoch_path, run_dir/"history.csv")
    log_epoch(
        run, epoch, result, selection_metric=config["selection_metric"],
        epoch_seconds=result["epoch_seconds"],
        extra_metrics={f"val_{config['validation_version']}/clean_v3/{key}": value
                       for key, value in clean_scores.items()},
    )
    print(result)
    del predictions

display(pd.DataFrame(history))
save_json(run_dir/"summary.json", {"best_epoch": best_epoch,
                                      "best_score": best_score,
                                      "selection_metric": config["selection_metric"]})
backup_files(backup_dir, run_dir/"summary.json")


Train:   0%|          | 0/7106 [00:01<?, ?it/s]

Predict:   0%|          | 0/1971 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.03668631307743534, 'epoch_seconds': 268.0572480869996, 'uk_val_loss': 0.020797996738985903, 'ru_val_loss': 0.02794066165126452, 'combined_val_loss': 0.022575898018608778, 'uk_comp_f1': 0.85303446, 'ru_comp_f1': 0.83830802, 'combined_comp_f1': 0.84932725, 'clean_uk_comp_f1': 0.85287717, 'clean_ru_comp_f1': 0.83830802, 'clean_combined_comp_f1': 0.84920939}


Train:   0%|          | 0/7102 [00:01<?, ?it/s]

Predict:   0%|          | 0/1971 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 0.018330236801497733, 'epoch_seconds': 271.0071286099992, 'uk_val_loss': 0.01781691240392489, 'ru_val_loss': 0.023508902733399697, 'combined_val_loss': 0.019233722098157897, 'uk_comp_f1': 0.87123523, 'ru_comp_f1': 0.86548805, 'combined_comp_f1': 0.86977786, 'clean_uk_comp_f1': 0.87089891, 'clean_ru_comp_f1': 0.86548805, 'clean_combined_comp_f1': 0.86952646}


Train:   0%|          | 0/7103 [00:01<?, ?it/s]

Predict:   0%|          | 0/1971 [00:00<?, ?it/s]

{'epoch': 3, 'train_loss': 0.014375098916653268, 'epoch_seconds': 270.0172503369995, 'uk_val_loss': 0.017644461301143784, 'ru_val_loss': 0.022892521806419208, 'combined_val_loss': 0.018950771145222815, 'uk_comp_f1': 0.87081166, 'ru_comp_f1': 0.86928735, 'combined_comp_f1': 0.8704255, 'clean_uk_comp_f1': 0.87071601, 'clean_ru_comp_f1': 0.869306, 'clean_combined_comp_f1': 0.8703587}


,epoch,train_loss,epoch_seconds,uk_val_loss,ru_val_loss,combined_val_loss,uk_comp_f1,ru_comp_f1,combined_comp_f1,clean_uk_comp_f1,clean_ru_comp_f1,clean_combined_comp_f1
0,1,0.036686,268.057248,0.020798,0.027941,0.022576,0.853034,0.838308,0.849327,0.852877,0.838308,0.849209
1,2,0.018330,271.007129,0.017817,0.023509,0.019234,0.871235,0.865488,0.869778,0.870899,0.865488,0.869526
2,3,0.014375,270.017250,0.017644,0.022893,0.018951,0.870812,0.869287,0.870425,0.870716,0.869306,0.870359


## 8. Validation comparison — Viterbi + identical LSTM postprocessing

This is a validation diagnostic, not Kaggle Public F1. Compare selected checkpoints with identical decoding and `expand=True, trim=True, empty=True`. Official metric is unchanged.

In [13]:
# Evaluate at most two candidates with identical validation decoding/cleanup,
# use viterbi_lines and clean_options from the LSTM
candidates = [run_dir / "best.pt", run_dir / f"epoch_{config['epochs']:02d}.pt"]
candidates = list(dict.fromkeys(candidates))

seen_epochs, comparison = set(), []
for candidate in candidates:
    ckpt = torch.load(candidate, map_location="cpu", weights_only=False)  # trusted local checkpoint
    if ckpt["epoch"] in seen_epochs:
        continue
    seen_epochs.add(ckpt["epoch"])
    assert ckpt["config"]["tokenizer"] == config["tokenizer"]
    model.load_state_dict(ckpt["model_state_dict"])
    current_predictions, _ = predict(model, val_loader, device, decoding="viterbi_lines")
    cleaned_scores = postprocessed_scores(val_records, current_predictions, CLEAN_OPTIONS)
    comparison.append({"checkpoint": candidate.name, "epoch": ckpt["epoch"],
                       **cleaned_scores})
    print(comparison[-1])
    del current_predictions, ckpt

comparison_df = pd.DataFrame(comparison).sort_values("uk_comp_f1", ascending=False)
display(comparison_df)
comparison_df.to_csv(run_dir/"viterbi_lines_clean_validation.csv", index=False)
backup_files(backup_dir, run_dir/"viterbi_lines_clean_validation.csv")
selected_checkpoint = run_dir / comparison_df.iloc[0]["checkpoint"]
print("Selected checkpoint:", selected_checkpoint)


Predict:   0%|          | 0/1971 [00:00<?, ?it/s]

{'checkpoint': 'best.pt', 'epoch': 2, 'uk_comp_f1': 0.87230397, 'ru_comp_f1': 0.89767462, 'combined_comp_f1': 0.87873615}


Predict:   0%|          | 0/1971 [00:00<?, ?it/s]

{'checkpoint': 'epoch_03.pt', 'epoch': 3, 'uk_comp_f1': 0.87600189, 'ru_comp_f1': 0.90176903, 'combined_comp_f1': 0.88252512}


,checkpoint,epoch,uk_comp_f1,ru_comp_f1,combined_comp_f1
1,epoch_03.pt,3,0.876002,0.901769,0.882525
0,best.pt,2,0.872304,0.897675,0.878736


Selected checkpoint: /content/outputs/bigru_ws_20261112T181358Z_c5b76509/epoch_03.pt


## 9. Kaggle test submission with selected checkpoint

In [15]:
ckpt = torch.load(selected_checkpoint, map_location="cpu", weights_only=False)  # trusted file
model = StackedBiGRUTagger(**ckpt["model_config"]).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

data_dir = get_data_dir(ENVIRONMENT)
test = pd.read_csv(data_dir / "test.csv", usecols=["text_id", "text"])
# Whole-message inference, no truncation and no chunking.
test_records = prepare_inference_records(tokenizer, test)
test_loader = make_loader(test_records, pad_id=pad_id, batch_size=config["batch_size"],
                          seed=config["seed"], max_padded_tokens=config["max_padded_tokens"],
                          pin_memory=True)
test_predictions, _ = predict(model, test_loader, device, decoding="viterbi_lines")
from src.postprocessing import postprocess
submission = test[["text_id"]].copy()
submission["locations"] = [
    json.dumps(postprocess(row.text, test_predictions[row.text_id]["spans"], **CLEAN_OPTIONS),
               ensure_ascii=False)
    for row in test.itertuples(index=False)
]
submission_path = run_dir / f"submission_bigru_whitespace_{selected_checkpoint.stem}_viterbi_lines.csv"
submission.to_csv(submission_path, index=False)
save_json(run_dir/"submission_config.json", {
    "checkpoint": str(selected_checkpoint), "epoch": int(ckpt["epoch"]),
    "tokenizer": str(tokenizer_path), "tokenizer_sha256": file_sha256(tokenizer_path),
    "decoding": "viterbi_lines", "postprocessing": CLEAN_OPTIONS,
    "full_messages": True, "model": "StackedBiGRUTagger",
})
backup_files(backup_dir, submission_path, run_dir/"submission_config.json")
display(submission.head())
print("Submission:", submission_path)


Predict:   0%|          | 0/6 [00:00<?, ?it/s]

,text_id,locations
0,0,"[""Київ""]"
1,1,"[""Києві"", ""Шулявського шляхопроводу"", ""Шулявсь..."
2,2,"[""Гоголеве"", ""Миргородського району"", ""Полтавс..."
3,3,[]
4,4,"[""проспекті Академіка Глушкова""]"


Submission: /content/outputs/bigru_ws_20261112T181358Z_c5b76509/submission_bigru_whitespace_epoch_03_viterbi_lines.csv


## 10. Saving results for ensemble

In [16]:
import json
from numbers import Integral
from pathlib import Path

def _prediction_rows(records, predictions):
    ids = [record["text_id"] for record in records]
    if not ids or len(ids) != len(set(ids)):
        raise ValueError("Records must be nonempty and have unique IDs")
    if set(ids) != set(predictions):
        raise ValueError("Missing or extra prediction IDs")

    rows = []
    for record in records:
        text = record["text"]
        if not isinstance(text, str):
            raise ValueError("Records must contain original text strings")
        spans = []
        for start, end in predictions[record["text_id"]]["spans"]:
            if (isinstance(start, bool) or isinstance(end, bool)
                    or not isinstance(start, Integral) or not isinstance(end, Integral)
                    or not 0 <= start < end <= len(text)):
                raise ValueError(f"Invalid span for {record['text_id']!r}")
            spans.append([int(start), int(end)])

        text_id = record["text_id"]
        if isinstance(text_id, Integral) and not isinstance(text_id, bool):
            text_id = int(text_id)
        elif not isinstance(text_id, str):
            raise ValueError("IDs must be strings or integers")
        rows.append({"text_id": text_id, "text": text, "spans": spans})
    return rows


def export_predictions(output_dir, checkpoint_path, model, predict_fn,
                       val_records, val_loader, test_records, test_loader):
    """Load a trusted local checkpoint and export both splits to a NEW folder.

    predict_fn must use the supplied model and must not reload weights, train,
    clean predictions, or change text/offsets. Existing output folders are refused.
    Matching architecture and tokenizer must already be prepared by the notebook.
    """
    import torch

    output_dir = Path(output_dir)
    if output_dir.exists():
        raise FileExistsError(f"Choose a new run folder: {output_dir}")

    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    print("Checkpoint:", checkpoint_path, "epoch:", checkpoint.get("epoch"))
    del checkpoint

    with torch.no_grad():
        val_predictions, _ = predict_fn(model, val_loader)
        val_rows = _prediction_rows(val_records, val_predictions)
        del val_predictions
        test_predictions, _ = predict_fn(model, test_loader)
        test_rows = _prediction_rows(test_records, test_predictions)
        del test_predictions

    # Validate both splits before creating any output files.
    output_dir.mkdir(parents=True, exist_ok=False)
    for filename, rows in [("val.jsonl", val_rows), ("test.jsonl", test_rows)]:
        with (output_dir / filename).open("x", encoding="utf-8") as file:
            for row in rows:
                file.write(json.dumps(row, ensure_ascii=False) + "\n")
        print(filename, len(rows), "texts")
    return output_dir

In [21]:
from datetime import datetime, timezone

# Use the model and loaders already prepared above
def predict_for_ensemble(model, loader):
    return predict(model, loader, device, decoding="viterbi_lines")

output_dir = (
    root / "outputs" / "ensemble_exports"
    / f"bigru_ensemble_{datetime.now(timezone.utc):%Y%m%d_%H%M%S_%f}"
)

export_predictions(
    output_dir=output_dir,
    checkpoint_path=selected_checkpoint,
    model=model,
    predict_fn=predict_for_ensemble,
    val_records=val_records,
    val_loader=val_loader,
    test_records=test_records,
    test_loader=test_loader,
)

print("Export completed:", output_dir)

Checkpoint: /content/outputs/bigru_ws_20261112T181358Z_c5b76509/epoch_03.pt epoch: 3


Predict:   0%|          | 0/1971 [00:00<?, ?it/s]

Predict:   0%|          | 0/6 [00:00<?, ?it/s]

val.jsonl 250867 texts
test.jsonl 477 texts
Export completed: /content/outputs/ensemble_exports/bigru_ensemble_20261112_184445_282981


In [22]:
import json

for filename, expected_count in [
    ("val.jsonl", len(val_records)),
    ("test.jsonl", len(test_records)),
]:
    filepath = output_dir / filename

    with filepath.open("r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]

    print(filename, ", Rows:", len(rows), ", Size MB:", round(filepath.stat().st_size / 1e6, 2))

    print("Example:", rows[0])

val.jsonl , Rows: 250867 , Size MB: 62.09
Example: {'text_id': 'uk_7', 'text': 'Але про присутність іншого, неідентифікованого літака, не говориться ні слова!', 'spans': []}
test.jsonl , Rows: 477 , Size MB: 0.37
Example: {'text_id': 0, 'text': '❗️Кількість поранених зросла до трьох, – Кличко\n\nКиїв — пряма трансляція 👈', 'spans': [[49, 53]]}


In [19]:
# Save the exported folder to Google Drive

drive_export_root = (
    Path("/content/drive/MyDrive/ner_results/shared")
    / "bigru"
)

drive_export_root.mkdir(parents=True, exist_ok=True)

drive_output_dir = drive_export_root / output_dir.name

shutil.copytree(output_dir, drive_output_dir)

# Save model checkpoint and tokenizer too
shutil.copy2(
    checkpoint_path,
    drive_output_dir / "checkpoint.pt"
)

shutil.copy2(
    tokenizer_path,
    drive_output_dir / "tokenizer.json"
)

# Verify backup
for filename in [
    "val.jsonl",
    "test.jsonl",
    "checkpoint.pt",
    "tokenizer.json",
]:
    assert (drive_output_dir / filename).is_file()

print("Saved to Google Drive:")
print(drive_output_dir)

Saved to Google Drive:
/content/drive/MyDrive/ner_results/shared/bigru/bigru_ensemble_20261112_183924


## 11. Finish W&B and export small results archive

In [ ]:
for row in comparison:
    prefix = f"evaluation/viterbi_lines_clean/epoch_{row['epoch']:02d}"
    for metric in ("uk_comp_f1", "ru_comp_f1", "combined_comp_f1"):
        run.summary[f"{prefix}/{metric}"] = row[metric]
run.summary["submission/selected_checkpoint"] = selected_checkpoint.name
run.summary["submission/path"] = str(submission_path)
finish_run(run, best_epoch=best_epoch, best_score=best_score,
           selection_metric=config["selection_metric"], checkpoint_path=best_path,
           upload_checkpoint=config["upload_checkpoint"])

